# 🧠 CerebroSpheroid V4: 1,000-Step V4-Small Training on Kaggle

### **Architecture & Enhancements in V4-Small:**
1. **Modern Llama 32,000 Vocabulary**: Byte-fallback BPE with atomic single-digit arithmetic (`0`-`9`) and zero OOV tokens (saving 35M parameters: 127M total).
2. **Causal Temporal Sensory Ingestion**: Exponential recency decay + phase velocity momentum vectors replacing order-erasing centroid pooling.
3. **18.6x Accelerated SpheroidOptimizer**: Persistent power iteration spectral radius guard (<1.5 ms) and decoupled row-wise RMS token momentum (<0.5 ms).
4. **Reasoning-Focused Mix**: Multi-step instruction and math reasoning (`OpenHermes=0.45, LMSYS=0.35, GSM8K=0.15, Cognitive=0.05`).

---
### ⚡ Kaggle Settings:
- **Accelerator**: Select **GPU T4 x2** (Dual GPU) or **GPU P100**
- **Internet**: Toggle **Internet ON**
- **Persistence**: Variables and Files

## Step 1: GPU Hardware Check

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.1f} GB VRAM)")

## Step 2: Clone / Pull Latest Repository

In [ ]:
import os
%cd /kaggle/working
if not os.path.exists("reasoning_agent"):
    !git clone https://github.com/AjitKumar-git/reasoning_agent.git
%cd /kaggle/working/reasoning_agent
!git fetch origin main
!git reset --hard origin/main
!pip install -q pyarrow fastparquet transformers datasets sentencepiece

## Step 3: Run Acceptance Verification Tests

In [ ]:
!python3 -u training/test_v4_architecture.py
!python3 -u training/test_spheroid_optimizer.py

## Step 4: Run 1,000-Step Training (V4 Small, SpheroidOptimizer)

In [ ]:
# Launch Dual T4 GPU Training with Cloud Sync and Elastic Window
!torchrun --nproc_per_node=2 training/train_v4_full.py --arch v4 --preset v4_small --tokenizer llama --mix "openhermes=0.45,lmsys=0.35,gsm8k=0.15,cognitive=0.05" --optimizer spheroid --cloud_sync --elastic_window --total_window 1536 --batch_size 4 --grad_accum 8 --lr 0.002 --max_steps 50000 --max_hours 8.5 --eval_window 128 --eval_interval 200 --save_every 100 --save_prefix cerebro_spheroid_v4_small_kaggle


## Step 5: Package Checkpoints for Download

In [ ]:
import os, glob
!mkdir -p /kaggle/working/checkpoints_out
!cp -r checkpoints/* /kaggle/working/checkpoints_out/ 2>/dev/null || true
!zip -r /kaggle/working/cerebro_spheroid_v4_small_1000steps.zip /kaggle/working/checkpoints_out
!ls -lh /kaggle/working/*.zip
print("\nCheckpoints successfully archived to /kaggle/working/cerebro_spheroid_v4_small_1000steps.zip!")
print("Download it directly from the right-hand Kaggle Output panel.")